# 🔍 Step 2 — Food Image Recognition (CLIP Model)
### Multimodal Food Recognition & Nutrition Assistant

**Goal:** Given a food photo, identify what the food is using OpenAI's CLIP model.

**How CLIP works:**
```
Food Photo ──► Image Encoder ──► Image Embedding ─┐
                                                    ├──► Similarity Score ──► Top Food Match
Food Labels ──► Text Encoder  ──►  Text Embedding ─┘
```

CLIP is a zero-shot model — it does NOT need to be trained on food images.
It matches images to text labels using shared embeddings.

**Input :** Any food photo (URL or uploaded file)
**Output:** Top-5 predicted food names + confidence scores

## ⚙️ 2.1 — Install Dependencies

In [ ]:
!pip install -q transformers torch torchvision Pillow requests
print('✅ Dependencies installed')

## 📦 2.2 — Import Libraries & Mount Drive

In [ ]:
import torch
import requests
import numpy as np
import pandas as pd
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from transformers import CLIPProcessor, CLIPModel
from google.colab import drive, files
import warnings
warnings.filterwarnings('ignore')

drive.mount('/content/drive')

# ─── CONFIGURE YOUR PATH ─────────────────────────────────────────────────
BASE       = '/content/drive/MyDrive/'                    # Same as Step 1
OUTPUT_DIR = BASE + 'nutrition_assistant_outputs/'
# ─────────────────────────────────────────────────────────────────────────

# Check GPU availability
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Libraries ready')
print(f'🖥️  Device: {DEVICE} ({"GPU available!" if DEVICE == "cuda" else "no GPU — will use CPU, slightly slower"})')

## 🤖 2.3 — Load CLIP Model

In [ ]:
print('Loading CLIP model... (first run downloads ~600MB, cached after)')

MODEL_NAME = 'openai/clip-vit-base-patch32'
model      = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE)
processor  = CLIPProcessor.from_pretrained(MODEL_NAME)

model.eval()  # Set to inference mode
print('✅ CLIP model ready')

## 🏷️ 2.4 — Build the Food Label Vocabulary

In [ ]:
# These are the food categories CLIP will try to match against.
# We combine: (a) common dish names, (b) categories from your food_category.csv
# The more specific these labels, the better CLIP performs.

FOOD_LABELS = [
    # Cooked dishes
    'pizza', 'burger', 'hamburger', 'sandwich', 'hot dog', 'sushi',
    'fried chicken', 'grilled chicken', 'chicken curry', 'butter chicken',
    'biryani', 'fried rice', 'noodles', 'pasta', 'spaghetti', 'lasagna',
    'tacos', 'burrito', 'quesadilla', 'nachos', 'fish and chips',
    'steak', 'grilled fish', 'shrimp', 'soup', 'stew', 'omelette',
    'pancakes', 'waffles', 'french toast', 'scrambled eggs', 'boiled eggs',
    'idli', 'dosa', 'samosa', 'dal', 'palak paneer', 'paneer tikka',
    'roti', 'naan', 'paratha', 'upma', 'poha',
    # Snacks & fast food
    'french fries', 'onion rings', 'popcorn', 'chips', 'spring rolls',
    'dim sum', 'dumplings', 'kebab',
    # Fruits
    'apple', 'banana', 'orange', 'mango', 'grapes', 'strawberry',
    'watermelon', 'pineapple', 'avocado', 'blueberries', 'peach',
    # Vegetables
    'salad', 'broccoli', 'carrot', 'tomato', 'cucumber', 'corn',
    'spinach', 'mushroom', 'bell pepper', 'cauliflower', 'potato',
    # Dairy & Bakery
    'ice cream', 'cake', 'cupcake', 'donut', 'croissant', 'muffin',
    'bread', 'toast', 'cheese', 'yogurt', 'butter',
    # Drinks
    'coffee', 'tea', 'juice', 'smoothie', 'milkshake', 'beer', 'wine',
    # Grains & Legumes
    'rice', 'oatmeal', 'cereal', 'lentils', 'chickpeas', 'beans'
]

# Format labels as natural language prompts — this improves CLIP accuracy
LABEL_PROMPTS = [f'a photo of {label}' for label in FOOD_LABELS]

print(f'✅ Food vocabulary: {len(FOOD_LABELS)} food categories')
print(f'   Sample labels: {FOOD_LABELS[:8]}')

## ⚡ 2.5 — Pre-compute Text Embeddings (Speed Optimisation)

In [ ]:
# Pre-compute text embeddings once and reuse for every image query.
# This makes prediction fast (no recomputing labels each time).

with torch.no_grad():
    text_inputs  = processor(text=LABEL_PROMPTS, return_tensors='pt',
                             padding=True, truncation=True).to(DEVICE)
    text_embeds  = model.get_text_features(**text_inputs)
    text_embeds  = text_embeds / text_embeds.norm(dim=-1, keepdim=True)  # Normalise

print(f'✅ Text embeddings pre-computed: shape {text_embeds.shape}')

## 🔧 2.6 — Core Prediction Function

In [ ]:
def load_image(source):
    """
    Load a PIL Image from either:
    - A URL string (e.g. 'https://...')
    - A local file path (e.g. '/content/drive/MyDrive/apple.jpg')
    """
    if isinstance(source, str) and source.startswith('http'):
        response = requests.get(source, timeout=10)
        return Image.open(BytesIO(response.content)).convert('RGB')
    return Image.open(source).convert('RGB')


def predict_food(image_source, top_n=5):
    """
    Identify the food in an image using CLIP.

    Args:
        image_source : URL string or local file path
        top_n        : number of top predictions to return

    Returns:
        results (list of dicts): [{'food': str, 'confidence': float}, ...]
        image   (PIL.Image)    : loaded image for display
    """
    image = load_image(image_source)

    with torch.no_grad():
        image_inputs = processor(images=image, return_tensors='pt').to(DEVICE)
        image_embeds = model.get_image_features(**image_inputs)
        image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)

        # Cosine similarity between image and all food text labels
        similarity   = (image_embeds @ text_embeds.T).squeeze(0)
        probs        = similarity.softmax(dim=-1).cpu().numpy()

    top_indices = probs.argsort()[::-1][:top_n]
    results = [
        {'food': FOOD_LABELS[i], 'confidence': float(probs[i])}
        for i in top_indices
    ]
    return results, image


def display_prediction(image_source, top_n=5):
    """
    Predict food and display the image alongside a bar chart of confidence scores.
    """
    results, image = predict_food(image_source, top_n=top_n)

    foods  = [r['food'] for r in results]
    scores = [r['confidence'] * 100 for r in results]
    colors = ['#2ecc71' if i == 0 else '#95a5a6' for i in range(len(foods))]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.patch.set_facecolor('#1a1a2e')

    # Left: food image
    axes[0].imshow(image)
    axes[0].axis('off')
    axes[0].set_title('Input Image', color='white', fontsize=13, pad=10)

    # Right: horizontal bar chart
    axes[1].set_facecolor('#16213e')
    bars = axes[1].barh(foods[::-1], scores[::-1], color=colors[::-1],
                        edgecolor='none', height=0.6)
    axes[1].set_xlabel('Confidence (%)', color='white', fontsize=11)
    axes[1].set_title('CLIP Predictions', color='white', fontsize=13, pad=10)
    axes[1].tick_params(colors='white')
    axes[1].spines[['top','right','left','bottom']].set_visible(False)
    for spine in axes[1].spines.values():
        spine.set_edgecolor('#333')

    # Add value labels on bars
    for bar, score in zip(bars, scores[::-1]):
        axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                     f'{score:.1f}%', va='center', color='white', fontsize=10)

    axes[1].set_xlim(0, max(scores) * 1.25)
    plt.tight_layout()
    plt.show()

    print(f'\n🏆 Top prediction: "{results[0]["food"].upper()}" ({results[0]["confidence"]*100:.1f}% confidence)')
    return results


print('✅ Prediction functions ready')

## 🧪 2.7 — Test with Sample Images (URL)

In [ ]:
# Test 1: Pizza
print('═' * 50)
print('TEST 1: Pizza')
print('═' * 50)
results = display_prediction(
    'https://upload.wikimedia.org/wikipedia/commons/a/a3/Eq_it-na_pizza-margherita_sep2005_sml.jpg'
)

In [ ]:
# Test 2: Banana
print('═' * 50)
print('TEST 2: Banana')
print('═' * 50)
results = display_prediction(
    'https://upload.wikimedia.org/wikipedia/commons/8/8a/Banana-Fruit.jpg'
)

## 📤 2.8 — Test with Your Own Uploaded Image

In [ ]:
# Upload any food photo from your computer
print('Select a food image to upload...')
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    filepath = f'/content/{filename}'
    print(f'\nRunning prediction on: {filename}')
    results = display_prediction(filepath)
else:
    print('No file uploaded.')

## 🔗 2.9 — Connect CLIP Output to Step 1 Nutrition Lookup

In [ ]:
# Load the master nutrition table built in Step 1
nutrition_wide = pd.read_csv(OUTPUT_DIR + 'master_nutrition.csv')

def lookup_nutrition(food_query, top_n=3):
    """Search master_nutrition.csv by food name."""
    query    = food_query.lower().strip()
    keywords = query.split()
    mask     = nutrition_wide['food_name_clean'].str.contains(keywords[0], na=False)
    for kw in keywords[1:]:
        mask &= nutrition_wide['food_name_clean'].str.contains(kw, na=False)
    results = nutrition_wide[mask]
    if results.empty:
        mask    = nutrition_wide['food_name_clean'].str.contains(keywords[0], na=False)
        results = nutrition_wide[mask]
    return results.head(top_n)


def image_to_nutrition(image_source):
    """
    Full pipeline: image → CLIP prediction → nutrition lookup.

    Args:
        image_source: URL or file path

    Returns:
        predicted_food (str)   : top CLIP prediction
        nutrition_df   (df)    : matched nutrition rows
        all_predictions (list) : full CLIP top-5 results
    """
    print('🔍 Running CLIP image recognition...')
    predictions, image = predict_food(image_source, top_n=5)
    predicted_food     = predictions[0]['food']
    confidence         = predictions[0]['confidence'] * 100

    print(f'✅ Predicted: "{predicted_food}" ({confidence:.1f}% confidence)')
    print(f'🔎 Looking up nutrition data...')

    nutrition_df = lookup_nutrition(predicted_food)

    if nutrition_df.empty:
        print(f'⚠️  No exact USDA match for "{predicted_food}". Trying fallback...')
        # Try with first keyword
        nutrition_df = lookup_nutrition(predicted_food.split()[0])

    return predicted_food, nutrition_df, predictions


print('✅ Pipeline bridge function ready')

## 🧪 2.10 — Test the Full Image → Nutrition Pipeline

In [ ]:
# Test the end-to-end pipeline with a sample image
TEST_IMAGE = 'https://upload.wikimedia.org/wikipedia/commons/a/a3/Eq_it-na_pizza-margherita_sep2005_sml.jpg'

food_name, nutrition_df, all_preds = image_to_nutrition(TEST_IMAGE)

print('\n── Nutrition Matches from USDA Database ──')
if not nutrition_df.empty:
    cols = ['food_name', 'category_name', 'calories_kcal', 'protein_g',
            'fat_g', 'carbs_g', 'sodium_mg']
    available_cols = [c for c in cols if c in nutrition_df.columns]
    display(nutrition_df[available_cols])
else:
    print('No USDA match found. Consider adding more keywords to FOOD_LABELS.')

## 💾 2.11 — Save the Model Pipeline State

In [ ]:
import pickle

# Save food labels so Step 4 can reuse them without reloading CLIP
pipeline_state = {
    'food_labels'  : FOOD_LABELS,
    'label_prompts': LABEL_PROMPTS,
    'model_name'   : MODEL_NAME,
    'device'       : DEVICE
}

with open(OUTPUT_DIR + 'clip_pipeline_state.pkl', 'wb') as f:
    pickle.dump(pipeline_state, f)

print('✅ Pipeline state saved to:', OUTPUT_DIR + 'clip_pipeline_state.pkl')

## ✅ Step 2 Complete!

| Capability | Status |
|---|---|
| CLIP model loaded | ✅ |
| Food label vocabulary (90 categories) | ✅ |
| Text embeddings pre-computed | ✅ |
| `predict_food(image)` function | ✅ |
| `image_to_nutrition(image)` pipeline | ✅ |
| Confidence bar chart visualisation | ✅ |

**Key functions for later steps:**
- `predict_food(image_source)` → top-5 food predictions with confidence
- `image_to_nutrition(image_source)` → full pipeline: image → food name → nutrition data

---
**➡️ Next: Step 3 — Recipe Matching Engine (RAG with FAISS)**